# 🚀 Renest × ComfyUI One-Click Restore & Launch

**Only named frameworks: run once on a cloud GPU, pack and restore verified dependencies and assets.**

Enter a **Hand-off code** provided by the community, sample galleries, or your team. This notebook verifies the environment, restores dependencies and weights, and launches the ComfyUI WebUI with a public access link.

In [ ]:
# @title 1. Restore ComfyUI Environment (Enter Hand-off Code and Run) 🚀
# @markdown Enter your **Hand-off code** below:
HANDOFF_CODE = "PASTE_YOUR_HANDOFF_CODE_HERE" # @param {type:"string"}
WORKSPACE = "/content/ComfyUI" # @param {type:"string"}

import json
import os
import subprocess
import sys

code = HANDOFF_CODE.strip()
assert code and code != "PASTE_YOUR_HANDOFF_CODE_HERE", "❌ Please provide a valid Hand-off code!"

# Resolve Hand-off code into grant file
grant_file = "/content/grant.json"
if os.path.isfile(code):
    grant_arg = code
elif code.startswith("{"):
    with open(grant_file, "w", encoding="utf-8") as f:
        f.write(code)
    grant_arg = grant_file
else:
    envelope = {
        "grant_version": "2",
        "grant_id": code,
        "origin": "https://api.renest.ai"
    }
    with open(grant_file, "w", encoding="utf-8") as f:
        json.dump(envelope, f)
    grant_arg = grant_file

print("📦 Installing / Updating Renest CLI...")
install_res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "renest"])
if install_res.returncode != 0:
    raise RuntimeError(f"Failed to install renest CLI (exit code {install_res.returncode})")

print("🔄 Running renest restore (health check -> asset transfer -> dependency resolution -> launch and render verification)...")
restore_res = subprocess.run([
    "renest", "restore",
    "--grant", grant_arg,
    "--dir", WORKSPACE,
    "--yes"
])

if restore_res.returncode != 0:
    raise RuntimeError(f"Restore failed with exit code {restore_res.returncode}. See above logs for details.")

print("✅ Restore completed successfully! Proceed to Step 2 to launch WebUI.")


In [ ]:
# @title 2. Launch ComfyUI WebUI & Public Tunnel 🎨
# @markdown Starts ComfyUI and creates a public web access link via Cloudflare tunnel.

import os
import subprocess
import sys

# 1. Verify Step 1 completed successfully before launching
start_facts = os.path.join(WORKSPACE, ".renest", "start.json")
venv_py = os.path.join(WORKSPACE, ".venv", "bin", "python")

if not (os.path.isdir(WORKSPACE) and os.path.isfile(start_facts) and os.path.isfile(venv_py)):
    raise RuntimeError(
        f"❌ Workspace '{WORKSPACE}' does not contain a verified restored environment.\n"
        "Please run Step 1 (Restore) first and ensure it finishes with exit code 0."
    )

# 2. Ensure supervisor runner is present (auto-provisioned if running on fresh Colab)
helper_script = "/tmp/renest_colab.py"
if not os.path.isfile(helper_script):
    with open(helper_script, "w", encoding="utf-8") as f:
        f.write('#!/usr/bin/env python3\n"""Renest Colab Runner: Launches ComfyUI and creates a public WebUI tunnel on Google Colab.\n\nRules & Architecture:\n1. Environment must be pre-restored via `renest restore`.\n2. Serves as a host supervisor calling `renest start` and running cloudflared tunnel.\n3. Does not modify renest CLI or server internals. Pure standard library implementation.\n4. User-facing text in English per project constitution.\n5. Clean lifecycle: guarantees termination of both child processes on any exit path.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport platform\nimport re\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.request\nfrom typing import Optional\n\n\nCLOUDFLARED_URLS = {\n    ("Linux", "x86_64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",\n    ("Linux", "aarch64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64",\n    ("Darwin", "x86_64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-amd64",\n    ("Darwin", "arm64"): "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-darwin-arm64",\n}\n\n\ndef ensure_cloudflared(dest_dir: str = "/tmp") -> str:\n    """Ensure cloudflared binary is available, downloading standalone binary if missing."""\n    system_bin = shutil.which("cloudflared")\n    if system_bin:\n        return system_bin\n\n    local_bin = os.path.join(dest_dir, "cloudflared")\n    if os.path.isfile(local_bin) and os.access(local_bin, os.X_OK):\n        return local_bin\n\n    os_type = platform.system()\n    machine = platform.machine()\n    download_url = CLOUDFLARED_URLS.get((os_type, machine))\n    if not download_url:\n        download_url = CLOUDFLARED_URLS[("Linux", "x86_64")]\n\n    print(f"[Renest Colab] Preparing Cloudflare tunnel binary ({os_type}/{machine})...", flush=True)\n    try:\n        urllib.request.urlretrieve(download_url, local_bin)\n        os.chmod(local_bin, 0o755)\n    except Exception as e:\n        raise RuntimeError(f"Failed to download cloudflared: {e}. Check network connection.") from e\n\n    return local_bin\n\n\ndef extract_tunnel_url(log_line: str) -> Optional[str]:\n    """Extract trycloudflare.com URL from cloudflared log line."""\n    match = re.search(r"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com", log_line)\n    if match:\n        return match.group(0)\n    return None\n\n\ndef wait_for_http_ready(port: int, timeout: float = 60.0) -> bool:\n    """Poll local HTTP port until ComfyUI answers HTTP 200, or timeout expires."""\n    deadline = time.monotonic() + timeout\n    endpoints = [f"http://127.0.0.1:{port}/system_stats", f"http://127.0.0.1:{port}/"]\n    while time.monotonic() < deadline:\n        for url in endpoints:\n            try:\n                req = urllib.request.Request(url, headers={"User-Agent": "Renest-Colab-HealthCheck/1.0"})\n                with urllib.request.urlopen(req, timeout=2.0) as resp:\n                    if resp.status == 200:\n                        return True\n            except (urllib.error.URLError, OSError):\n                pass\n        time.sleep(1.0)\n    return False\n\n\ndef _terminate_proc(proc: Optional[subprocess.Popen], timeout: float = 5.0) -> None:\n    """Terminate a subprocess and its entire process group, ensuring zero orphans.\n\n    Reaps direct child processes with wait() to avoid zombie/defunct entries.\n    Raises RuntimeError if any process in the group remains alive after SIGKILL.\n    """\n    if proc is None:\n        return\n\n    pid = proc.pid\n    try:\n        pgid = os.getpgid(pid)\n    except (AttributeError, OSError, ProcessLookupError):\n        pgid = pid\n\n    def _group_alive() -> bool:\n        try:\n            os.killpg(pgid, 0)\n            return True\n        except (AttributeError, OSError, ProcessLookupError):\n            return proc.poll() is None\n\n    def _reap_proc() -> None:\n        try:\n            proc.wait(timeout=0.2)\n        except (subprocess.TimeoutExpired, OSError, AttributeError):\n            pass\n\n    if not _group_alive():\n        _reap_proc()\n        return\n\n    # 1. Try graceful SIGINT first to allow clean application teardown\n    try:\n        try:\n            os.killpg(pgid, signal.SIGINT)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.send_signal(signal.SIGINT)\n        deadline = time.monotonic() + 3.0\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.1)\n    except OSError:\n        pass\n\n    # 2. Fall back to SIGTERM\n    try:\n        try:\n            os.killpg(pgid, signal.SIGTERM)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.terminate()\n        deadline = time.monotonic() + timeout\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.1)\n    except OSError:\n        pass\n\n    # 3. Force SIGKILL on process group\n    try:\n        try:\n            os.killpg(pgid, signal.SIGKILL)\n        except (AttributeError, OSError, ProcessLookupError):\n            proc.kill()\n        deadline = time.monotonic() + 2.0\n        while time.monotonic() < deadline:\n            if not _group_alive():\n                _reap_proc()\n                return\n            time.sleep(0.05)\n    except OSError:\n        pass\n\n    _reap_proc()\n    if _group_alive():\n        raise RuntimeError(\n            f"Process teardown failed: process group {pgid} (PID {pid}) still active after SIGKILL"\n        )\n\n\ndef _cleanup_procs(app_proc: Optional[subprocess.Popen], tunnel_proc: Optional[subprocess.Popen]) -> None:\n    """Ensure both child processes are terminated and waited for, preventing orphan processes.\n\n    Raises RuntimeError if teardown fails to terminate either process group.\n    """\n    errors = []\n    for name, p, t in [("tunnel", tunnel_proc, 3.0), ("app", app_proc, 5.0)]:\n        try:\n            _terminate_proc(p, timeout=t)\n        except Exception as e:\n            errors.append(f"{name}: {e}")\n    if errors:\n        msg = f"Process teardown failed: {\'; \'.join(errors)}"\n        print(f"[Renest Colab] Error: {msg}", file=sys.stderr, flush=True)\n        raise RuntimeError(msg)\n\n\ndef launch_webui(\n    workspace: str,\n    port: int = 8188,\n    listen: str = "0.0.0.0",\n    cloudflared_path: Optional[str] = None,\n    timeout_tunnel: float = 30.0,\n    timeout_http: float = 60.0,\n) -> int:\n    """Launch `renest start` and cloudflared tunnel, continuously draining pipes and guarding lifecycle."""\n    if not os.path.isdir(workspace):\n        print(f"Error: Workspace directory does not exist: {workspace}", file=sys.stderr)\n        print("Please run `renest restore` first to restore the environment.", file=sys.stderr)\n        return 1\n\n    cf_bin = cloudflared_path or ensure_cloudflared()\n\n    app_proc: Optional[subprocess.Popen] = None\n    tunnel_proc: Optional[subprocess.Popen] = None\n\n    def _sig_handler(signum, frame):\n        raise KeyboardInterrupt()\n\n    prev_int = signal.signal(signal.SIGINT, _sig_handler)\n    prev_term = signal.signal(signal.SIGTERM, _sig_handler)\n\n    try:\n        print(f"[Renest Colab] Starting ComfyUI via `renest start` (port {port})...", flush=True)\n        start_cmd = [\n            "renest",\n            "start",\n            "--dir",\n            workspace,\n            "--listen",\n            listen,\n            "--port",\n            str(port),\n        ]\n\n        app_proc = subprocess.Popen(\n            start_cmd,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            text=True,\n            bufsize=1,\n            start_new_session=True,\n        )\n\n        tunnel_cmd = [\n            cf_bin,\n            "tunnel",\n            "--url",\n            f"http://127.0.0.1:{port}",\n        ]\n\n        print("[Renest Colab] Establishing Cloudflare tunnel...", flush=True)\n        tunnel_proc = subprocess.Popen(\n            tunnel_cmd,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            text=True,\n            bufsize=1,\n            start_new_session=True,\n        )\n\n        tunnel_url_holder: list[str] = []\n        tunnel_found_event = threading.Event()\n\n        # Continuous pipe draining threads to avoid OS 64KB pipe buffer deadlock\n        def drain_tunnel():\n            if not tunnel_proc or not tunnel_proc.stdout:\n                return\n            for line in tunnel_proc.stdout:\n                if not tunnel_found_event.is_set():\n                    url = extract_tunnel_url(line)\n                    if url:\n                        tunnel_url_holder.append(url)\n                        tunnel_found_event.set()\n\n        def drain_app():\n            if not app_proc or not app_proc.stdout:\n                return\n            for line in app_proc.stdout:\n                if any(k in line for k in ("Starting server", "Import times", "To see the GUI", "Traceback", "Error")):\n                    print(f"[ComfyUI] {line.strip()}", flush=True)\n\n        t_tunnel = threading.Thread(target=drain_tunnel, daemon=True)\n        t_app = threading.Thread(target=drain_app, daemon=True)\n        t_tunnel.start()\n        t_app.start()\n\n        # 1. Wait for tunnel URL\n        if not tunnel_found_event.wait(timeout=timeout_tunnel):\n            if tunnel_proc.poll() is not None:\n                print(f"Error: cloudflared exited prematurely (code {tunnel_proc.returncode}).", file=sys.stderr)\n                return tunnel_proc.returncode or 1\n            if app_proc.poll() is not None:\n                print(f"Error: renest start exited prematurely (code {app_proc.returncode}).", file=sys.stderr)\n                return app_proc.returncode or 1\n            print(f"Error: Timed out waiting for Cloudflare tunnel URL ({timeout_tunnel}s).", file=sys.stderr)\n            return 1\n\n        tunnel_url = tunnel_url_holder[0]\n\n        # 2. Health check ComfyUI HTTP endpoint: MUST succeed before announcing ready\n        print(f"[Renest Colab] Waiting for ComfyUI to answer on port {port}...", flush=True)\n        http_ready = wait_for_http_ready(port=port, timeout=timeout_http)\n        if not http_ready:\n            print(f"Error: ComfyUI HTTP health check timed out after {timeout_http}s.", file=sys.stderr)\n            if app_proc.poll() is not None:\n                print(f"Error: ComfyUI process terminated with exit code {app_proc.returncode}", file=sys.stderr)\n                return app_proc.returncode or 1\n            return 1\n\n        # 3. Both tunnel URL and HTTP health check passed: declare ready\n        banner = [\n            "",\n            "═" * 70,\n            "  🎉 ComfyUI WebUI is ready!",\n            f"  👉 Public WebUI URL: {tunnel_url}",\n            "     (Unauthenticated public tunnel via Cloudflare. All models loaded)",\n            "═" * 70,\n            "",\n        ]\n        print("\\n".join(banner), flush=True)\n\n        # 4. Supervise until interrupt or child process termination\n        while True:\n            if app_proc.poll() is not None:\n                print(f"\\n[Renest Colab] ComfyUI process exited (code {app_proc.returncode}).", file=sys.stderr)\n                return app_proc.returncode\n            if tunnel_proc.poll() is not None:\n                print(f"\\n[Renest Colab] cloudflared process exited (code {tunnel_proc.returncode}).", file=sys.stderr)\n                return tunnel_proc.returncode\n            time.sleep(1)\n\n    except KeyboardInterrupt:\n        print("\\n[Renest Colab] Shutting down services...", flush=True)\n        return 0\n    finally:\n        signal.signal(signal.SIGINT, prev_int)\n        signal.signal(signal.SIGTERM, prev_term)\n        # Guarantee both processes are cleaned up on any exit path\n        _cleanup_procs(app_proc, tunnel_proc)\n\n\ndef main(argv: Optional[list[str]] = None) -> int:\n    parser = argparse.ArgumentParser(\n        description="Renest Colab Helper: Launches ComfyUI and establishes public tunnel."\n    )\n    subparsers = parser.add_subparsers(dest="subcommand", required=True)\n\n    launch_p = subparsers.add_parser("launch", help="Launch ComfyUI and tunnel")\n    launch_p.add_argument(\n        "--dir",\n        default="/content/ComfyUI",\n        help="Workspace directory (default: /content/ComfyUI)",\n    )\n    launch_p.add_argument(\n        "--port",\n        type=int,\n        default=8188,\n        help="ComfyUI listen port (default: 8188)",\n    )\n    launch_p.add_argument(\n        "--listen",\n        default="0.0.0.0",\n        help="ComfyUI listen host (default: 0.0.0.0)",\n    )\n    launch_p.add_argument(\n        "--timeout-tunnel",\n        type=float,\n        default=30.0,\n        help="Max seconds to wait for tunnel URL (default: 30.0)",\n    )\n    launch_p.add_argument(\n        "--timeout-http",\n        type=float,\n        default=60.0,\n        help="Max seconds to wait for ComfyUI HTTP readiness (default: 60.0)",\n    )\n\n    args = parser.parse_args(argv)\n\n    try:\n        if args.subcommand == "launch":\n            return launch_webui(\n                workspace=args.dir,\n                port=args.port,\n                listen=args.listen,\n                timeout_tunnel=args.timeout_tunnel,\n                timeout_http=args.timeout_http,\n            )\n    except RuntimeError as e:\n        print(f"Error: {e}", file=sys.stderr)\n        return 1\n\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n')

print("🚀 Launching ComfyUI WebUI supervisor...")
res = subprocess.run([sys.executable, helper_script, "launch", "--dir", WORKSPACE])
if res.returncode != 0:
    raise RuntimeError(f"ComfyUI WebUI runner exited with code {res.returncode}")


---
### 💡 Next Steps: Save your modified workflows and models
Once you finish generating images or installing new custom nodes, run `renest pack` to store verified working dependencies and assets into your own nest.

In [ ]:
# @title 3. (Optional) Pack Environment into a New Nest 💾
!renest pack --dir "$WORKSPACE"
